# VideoMAE + Slot Attention Workbench

Clean working notebook for:

- controlled VideoMAE baselines
- VideoMAE + Slot experiments
- validation evaluation
- Slot diagnostics
- temporal/tubelet analysis
- visualization
- future K sweeps and larger-class experiments

Reusable implementation lives in `src/`.
This notebook should contain only configuration, calls, tables, plots, and interpretation.

In [1]:
from pathlib import Path
import sys


def find_repo_root():
    start = Path.cwd().resolve()

    candidates = [
        start,
        *start.parents,
    ]

    for candidate in candidates:
        if (
            (candidate / "src").is_dir()
            and
            (candidate / "notebooks").is_dir()
        ):
            return candidate

    raise RuntimeError(
        "Could not find repository root. "
        "Run the notebook from somewhere inside the repository."
    )


REPO_ROOT = find_repo_root()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(REPO_ROOT),
    )


print(
    "repository root:",
    REPO_ROOT,
)

repository root: /home/entezari/Documents/Research/slot-attention


In [2]:
import math

import pandas as pd
import torch
import torch.nn as nn

from src.checkpoints import (
    load_checkpoint_model_state,
)

from src.data import (
    build_dataloaders,
    build_ssv2_datasets,
    load_class_subset,
)

from src.diagnostics import (
    collect_slot_diagnostics,
    evaluate_dominant_slot_ablation,
    summarize_diagnostic_rows,
    videomae_temporal_statistics,
)

from src.evaluation import (
    evaluate,
    evaluate_slot_per_video_init,
)

from src.models import (
    build_baseline_model,
    build_slot_model,
    get_baseline_processor_name,
    get_slot_processor_name,
)

from src.utils import (
    seed_everything,
)

from src.visualization import (
    plot_slot_assignment_maps,
    plot_slot_tubelet_heatmap,
    plot_temporal_profiles,
    plot_video_frames,
    plot_winner_confidence_maps,
    select_examples_by_metric,
)


print(
    "clean repo imports OK"
)

clean repo imports OK


In [3]:
# ============================================================
# EXPERIMENT CONFIG
# ============================================================

MODEL_NAME = (
    "videomae_controlled"
)

NUM_CLASSES = 5
NUM_FRAMES = 16

NUM_SLOTS = 4
SLOT_DIM = 128
SLOT_ITERS = 3

TRAINING_SEED = 42

SLOT_EVAL_MODE = (
    "per_video_deterministic"
)

SLOT_EVAL_SEED = 12345

BATCH_SIZE = 2
NUM_WORKERS = 2

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


# ============================================================
# PATHS
#
# Change only these three for your machine / Colab / Kaggle.
# ============================================================

SPLITS_DIR = Path(
    "/PATH/TO/SPLITS"
)

CLIPS_DIR = Path(
    "/PATH/TO/SSV2/CLIPS"
)

CHECKPOINT_PATH = Path(
    "/PATH/TO/CORRECTED_K4/best.pt"
)


# ============================================================
# EVALUATION POLICY
# ============================================================

EVALUATE_TEST = False

TOPK_VALUES = [1]

In [4]:
seed_everything(
    TRAINING_SEED
)


processor_name = (
    get_slot_processor_name(
        MODEL_NAME,
        slot_eval_mode=SLOT_EVAL_MODE,
    )
)


print(
    "device:",
    DEVICE,
)

print(
    "model:",
    MODEL_NAME,
)

print(
    "classes:",
    NUM_CLASSES,
)

print(
    "frames:",
    NUM_FRAMES,
)

print(
    "slots:",
    NUM_SLOTS,
)

print(
    "slot dim:",
    SLOT_DIM,
)

print(
    "slot iterations:",
    SLOT_ITERS,
)

print(
    "training seed:",
    TRAINING_SEED,
)

print(
    "evaluation mode:",
    SLOT_EVAL_MODE,
)

print(
    "evaluation init seed:",
    SLOT_EVAL_SEED,
)

print(
    "processor:",
    processor_name,
)

print(
    "evaluate test:",
    EVALUATE_TEST,
)


assert (
    MODEL_NAME
    == "videomae_controlled"
)

assert NUM_FRAMES == 16
assert NUM_SLOTS >= 1

assert (
    processor_name
    ==
    "MCG-NJU/"
    "videomae-base-finetuned-kinetics"
)

assert (
    EVALUATE_TEST
    is False
)


print()
print(
    "✓ WORKBENCH CONFIGURATION OK"
)

device: cuda
model: videomae_controlled
classes: 5
frames: 16
slots: 4
slot dim: 128
slot iterations: 3
training seed: 42
evaluation mode: per_video_deterministic
evaluation init seed: 12345
processor: MCG-NJU/videomae-base-finetuned-kinetics
evaluate test: False

✓ WORKBENCH CONFIGURATION OK
